In [1]:
# import of modules
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import soundfile as sf
import math
import seaborn as sns
from scipy import signal
%matplotlib inline

In [ ]:
# setting of matplotlib
# font
plt.rcParams['font.family'] = 'Times New Roman'

# font of equation
plt.rcParams['mathtext.fontset'] = 'cm'

# direction of x and y-scales
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'

# showing of minor x and y-scales
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['ytick.minor.visible'] = True

# showing of scales in all directionsa
plt.rcParams["xtick.top"] = True
plt.rcParams["xtick.bottom"] = True
plt.rcParams["ytick.left"] = True
plt.rcParams["ytick.right"] = True

# font size
plt.rcParams["font.size"] = 30

In [3]:
# function of Fourier transform
def fourier_transform(data, samplerate):
    
    # preprocessing by window function
    N = len(data)
    window = np.hanning(N)
    input_data = data * window
    
    # fast Fourier transform
    spectrum = np.fft.fft(data)
    
    # calculation of amplitude spectrum
    amplitude = np.sqrt((spectrum.real**2) + (spectrum.imag**2)) / (N / 2)
    
    # generalization
    amplitude = 1 / (sum(window) / N) * amplitude
    
    # truncation at Nyquist frequency
    amplitude = amplitude[:int(len(amplitude)/2)]
    
    return amplitude**2

In [4]:
# High pass filter
def highpass_filter(x, samplerate, fp, fs, gpass, gstop):
    fn = samplerate / 2
    wp = fp / fn
    ws = fs / fn
    N, Wn = signal.buttord(wp, ws, gpass, gstop)
    b, a = signal.butter(N, Wn, "high")
    y = signal.filtfilt(b, a, x)
    return y

In [5]:
# Short time Fourier transform
def calc_stft(data, sample_number, samplerate):
    
    # matrix of stft
    stft = []
    
    for i in range(sample_number):
        b1 = i * ((len(data) - sample_number*2) // sample_number)
        b2 = b1 + sample_number*2
        power = fourier_transform(data[b1:b2], samplerate)
        stft.append(power)
        
    return np.array(stft)

In [ ]:
# choosing SNR (please unlock comment out)
SNR = 0
# SNR = -4
# SNR = -8
# SNR = -12
# SNR = -16
# SNR = -20

# choosing data type (please unlock comment out)
data_type = 'water_flow_125'
# data_type = 'gas-liquid_jet_0.15'
# data_type = 'gas-liquid_jet_0.17'
# data_type = 'gas-liquid_jet_0.20'

# filename
# filename1 = '../01_Original_Data/water_flow_125.wav'
filename1 = '"C:\Users\Casper4\Python\ueki\shibasaki\研究\water_flow\water_flow_125.wav"'
filename2 = '../01_Original_Data/' + data_type + '.wav'

# reading of wav file
data1, samplerate = sf.read(filename1)
data2, samplerate = sf.read(filename2)

# preprocessing by high pass filter
data1 = highpass_filter(data1[:2646000,0], samplerate, 1000, 900, 0.00001, 0.0001)
data2 = highpass_filter(data2[:2646000,0], samplerate, 1000, 900, 0.00001, 0.0001)

# amplify background noise according to SNR
data1_power = sum(data1**2) / len(data1)
data2_power = sum(data2**2) / len(data2)
data1 = data1 * np.sqrt((data2_power / 10 ** (SNR / 10)) / data1_power)

# normal sound -> water flow sound
# anomaly sound -> water flow sound + bubble jet flow sound
if data_type == 'water_flow_125':
    all_data = data1
else:
    all_data = data1 + data2

# segment length
data_length = 4410

# sample number
sample_number = 672

# size of picture (224 pixel × 224 pixel)
number = 224

for i in range(600):
    start = data_set * i
    finish = data_set * (i+1)
    # size of stft = sample_number × sample_number
    stft = calc_stft(all_data[start:finish], sample_number, samplerate)

    # resize into number × number
    # matix of compressed stft
    stft_comp = np.zeros((number, number))
    # extract the maximum value in each interval
    interval = int(sample_number / number)
    for j in range(number):
        for k in range(number):
            stft_comp[j][k] = np.max(stft[int(j*interval):int((j+1)*interval), int(k*interval):int((k+1)*interval)])

    # drawing and saving of heatmap
    dpi = 300
    fig = plt.figure(figsize=(number/dpi, number/dpi), dpi=dpi)
    sns.heatmap(np.rot90(stft_comp/np.max(stft_comp),k=1), cmap='jet', cbar=None, square=True)
    plt.axis('off')
    plt.subplots_adjust(left=0, right=1, bottom=0, top=1)
    plt.box(False)

    if data_type == 'water_flow_125':
        fig.savefig('../03_Extracted_data/SNR' + str(SNR) + '/01_Water_Flow_125/01_STFT/' + str(i) + '.png')
        plt.close()
    elif data_type == 'gas-liquid_jet_0.15':
        fig.savefig('../03_Extracted_data/SNR' + str(SNR) + '/02_Gas-Liquid_Jet_0.15/01_STFT/' + str(i) + '.png')
    elif data_type == 'gas-liquid_jet_0.17':
        fig.savefig('../03_Extracted_data/SNR' + str(SNR) + '/02_Gas-Liquid_Jet_0.17/01_STFT/' + str(i) + '.png')
    elif data_type == 'gas-liquid_jet_0.17':
        fig.savefig('../03_Extracted_data/SNR' + str(SNR) + '/02_Gas-Liquid_Jet_0.17/01_STFT/' + str(i) + '.png')
        plt.close()